# 🧠 บทที่ 3 — สอน AI ให้รู้จักปาล์ม (Train Model)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/A-Phrom-A/ssb-workshop/blob/main/03_train_model.ipynb)


> **สื่อการสอน Workshop:** การจำแนกพื้นที่ปลูกปาล์มน้ำมันด้วยภาพถ่ายดาวเทียมและ AI  
> ต่อจาก: [บทที่ 2 — เตรียม Tiles](02_prepare_tiles.ipynb)  
> ⚡ **ต้องใช้ GPU** — ไปที่เมนู Runtime → Change runtime type → GPU

---

## บทที่ 3 นี้ เราจะทำอะไร?

เมื่อเตรียม Tiles เสร็จแล้ว ถึงเวลา **"สอน AI"** ให้เรียนรู้ว่า  
pixel ไหนในภาพ SAR คือปาล์ม และ pixel ไหนไม่ใช่

![2.3.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/2.3.png)

ในบทนี้เราจะ:
1. ทำความเข้าใจ **Semantic Segmentation** — AI ระบายสี pixel
2. เรียนรู้ **SegFormer + ResNet-152** — โมเดลที่ใช้
3. **กำหนด Hyperparameters** — ตัวเลขที่ควบคุมการเรียนรู้
4. **Training!** — สอน AI จริง
5. **ประเมินผล** บน Hold-out data

---

## ทฤษฎี: Semantic Segmentation คืออะไร?

**Semantic Segmentation** = ให้ AI "ระบายสี" ทุก pixel ในภาพ  
บอกว่า pixel ไหนเป็นอะไร

![2.4.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/2.4.png)

### SegFormer + ResNet-152 คืออะไร?

โมเดลที่เราใช้ประกอบด้วย 2 ส่วนหลักที่ทำงานร่วมกัน:

![2.5.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/2.5.png)

> 💡 **Pre-trained weights:** โมเดลเริ่มต้นจากการ "ดูภาพทั่วไป" มาแล้วนับล้านภาพ (Pre-trained weights)  
> เราไม่ต้องสอนจากศูนย์ แค่ "สอนเพิ่ม" (fine-tune) ให้มันเชี่ยวชาญเรื่องปาล์ม

---

## ขั้นตอนที่ 1 — ติดตั้งและเตรียมเครื่องมือ

In [ ]:
# =====================================================================
# ⚠️ สำหรับรันบน Local Machine (เครื่องตัวเอง)
# =====================================================================
import os
BASE_DIR = r'C:\Users\kittisak\Downloads\ssb tutor\drive_structure\ch03_train_model'


In [ ]:
# =====================================================================
# ☁️ สำหรับรันบน Google Colab (คอมเมนต์ไว้ก่อน)
# =====================================================================
# from google.colab import drive
# drive.mount('/content/drive')
# BASE_DIR = '/content/drive/MyDrive/ssb_workshop/ch03_train_model'


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.1 — ติดตั้ง library (ต้องใช้ GPU version)
# ═══════════════════════════════════════════════════════════

!pip install geoai-py torch torchvision rasterio geopandas scikit-learn --quiet

import torch
print(f"✅ PyTorch: {torch.__version__}")
print(f"   CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("   ⚠️ ไม่พบ GPU — ไปที่ Runtime → Change runtime type → GPU")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1.2 — Import + กำหนด paths
# ═══════════════════════════════════════════════════════════

import os
import glob
import re
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import geopandas as gpd
import geoai
import logging

logging.basicConfig(level=logging.INFO, force=True)


In [ ]:
# ── Paths (ต่อจากบทที่ 2) ──
OUTPUT_DIR      = os.path.join(BASE_DIR, "output")
COMBINED_IMAGES = os.path.join(OUTPUT_DIR, "images")
COMBINED_LABELS = os.path.join(OUTPUT_DIR, "labels")
RASTER_DIR      = os.path.join(BASE_DIR, "data")
VECTOR_PATH     = os.path.join(BASE_DIR, "data", "ground_truth.geojson")

# ── นับ tiles ──
n_images = len(glob.glob(os.path.join(COMBINED_IMAGES, "*.tif")))
n_labels = len(glob.glob(os.path.join(COMBINED_LABELS, "*.tif")))
print(f"\n📊 Combined Dataset: {n_images:,} images, {n_labels:,} labels")
print("✅ พร้อม Training!")

---

## ขั้นตอนที่ 2 — กำหนด Hyperparameters

**Hyperparameters** = การตั้งค่าที่เรากำหนดก่อนเริ่มฝึก AI (คล้ายๆ การตั้งค่าความแรงของเตาอบก่อนทำขนม)

<!-- 🖼️ โครงสร้างสำหรับสร้างรูปประกอบ (ผู้สอนจะนำไปสร้างภาพจริงมาใส่แทน) -->
![2.6.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/2.6.png)

> 💡 Production code ใช้ 500 epochs — ใน Workshop เราใช้ **50 epochs** เพื่อประหยัดเวลา  ดังนั้น EARLY STOP จะถูกปรับลดลงมาให้เหลือ 5


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2.1 — กำหนด Hyperparameters
# ═══════════════════════════════════════════════════════════

ARCHITECTURE = "segformer"     # Decoder architecture
ENCODER_NAME = "resnet152"     # Encoder (pretrained)
NUM_CHANNELS = 2               # VV + VH
NUM_CLASSES  = 2               # 0=background, 1=palm
BATCH_SIZE   = 8               # tiles ต่อ batch
LEARNING_RATE = 0.0005         # ความเร็วเรียนรู้
NUM_EPOCHS   = 50              # จำนวนรอบ (production=500, workshop=50)
VAL_SPLIT    = 0.1             # 10% สำหรับ validation
EARLY_STOP   = 5              # หยุดถ้า val loss ไม่ดีขึ้นใน 30 epochs

# ── โฟลเดอร์ output สำหรับ model ──
MODEL_OUTPUT = os.path.join(OUTPUT_DIR, "segformer_models")

print("📋 Hyperparameters พร้อมแล้ว! โมเดลจะถูกบันทึกที่:")
print(f"   {MODEL_OUTPUT}")

---

## ขั้นตอนที่ 3 — Training!

### ข้อควรทราบ: Monkeypatch drop_last

เนื่องจาก `geoai` ใช้ Batch Normalization ซึ่งต้องการ batch ที่มีมากกว่า 1 รูป  
แต่ batch สุดท้ายตอนดึงข้อมูลอาจเหลือเศษแค่ 1 รูป ทำให้เกิด error  
เราจึงต้องใส่โค้ดบรรทัดพิเศษเพื่อบอกให้ "ทิ้งเศษ batch สุดท้าย" (drop_last=True)

> ⏱️ **เวลาที่ใช้:** ~30-60 นาทีต่อ 50 epochs (ขึ้นกับ GPU และจำนวน tiles)  
> *ระหว่างนี้เราจะไปดื่มกาแฟรอได้เลย!*

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3.1 — Training!
# ═══════════════════════════════════════════════════════════
# สังเกตค่า loss ที่ลดลงเรื่อยๆ แปลว่า AI กำลังเก่งขึ้น

import functools
import geoai.train as _geoai_train_module
from torch.utils.data import DataLoader as _TorchDataLoader

# ── Monkeypatch: ทิ้ง batch สุดท้ายที่เล็กเกินไป ──
_original_dataloader_cls = _geoai_train_module.DataLoader
_geoai_train_module.DataLoader = functools.partial(_TorchDataLoader, drop_last=True)

try:
    geoai.train_segmentation_model(
        images_dir=COMBINED_IMAGES,       # โฟลเดอร์ tiles ภาพ SAR
        labels_dir=COMBINED_LABELS,       # โฟลเดอร์ tiles ป้ายคำตอบ (0/1)
        output_dir=MODEL_OUTPUT,          # โฟลเดอร์เก็บ model (.pth)
        architecture=ARCHITECTURE,        # segformer
        encoder_name=ENCODER_NAME,        # resnet152
        encoder_weights="imagenet",       # เริ่มจาก weights ที่ฉลาดอยู่แล้ว
        num_channels=NUM_CHANNELS,        # 2 (VV + VH)
        num_classes=NUM_CLASSES,           # 2 (0=ไม่ใช่, 1=ปาล์ม)
        batch_size=BATCH_SIZE,            # 8
        num_epochs=NUM_EPOCHS,            # 50
        learning_rate=LEARNING_RATE,      # 0.0005
        val_split=VAL_SPLIT,              # 0.1
        early_stopping_patience=EARLY_STOP,  # 5
        save_best_only=False,             # เก็บ checkpoint ทุก 10 epochs
        verbose=True,
    )
finally:
    # ── คืนค่า DataLoader เดิม เมื่อ train เสร็จ ──
    _geoai_train_module.DataLoader = _original_dataloader_cls

print("\n✅ Training เสร็จสิ้น!")
print(f"   Model บันทึกที่: {MODEL_OUTPUT}")

---

## ทฤษฎี: Early Stopping — ทำไมต้องหยุดก่อนครบ?

ถ้าสอน AI นานเกินไป จะเกิดปัญหา **Overfitting** — AI "ท่องจำคำตอบ" จากแบบฝึกหัด แทนที่จะเข้าใจจริงๆ ทำให้พอไปทำข้อสอบจริงแล้วสอบตก

![2.7.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/2.7.png)

**Early Stopping** = ระบบจะจับตาดู Validation Loss ถ้าค่าไม่ลดลงติดต่อกัน 5 epochs (ตามค่า early_stopping) ระบบจะหยุด Train ทันที เพื่อป้องกันการท่องจำ

---

## ขั้นตอนที่ 4 — ประเมินผลบน Hold-out (ข้อสอบจริง)

การวัดผลที่น่าเชื่อถือที่สุด คือการให้ AI ลองทำนายบนภาพของเดือน **M05** ซึ่งเป็นข้อมูลที่มันไม่เคยเห็นเลยแม้แต่น้อย (Hold-out Set)

### ตัวชี้วัด (Metrics) ที่ใช้ให้คะแนน AI:

![2.8.png](https://raw.githubusercontent.com/A-Phrom-A/ssb-workshop/main/img/2.8.png)


In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4.1 — ฟังก์ชัน evaluate_on_holdout
# ═══════════════════════════════════════════════════════════
# ฟังก์ชันนี้จะนำโมเดลไปทำนายบนภาพใหญ่ทั้งภาพ
# แล้วนำผลมาเทียบกับ GeoJSON โดยอัตโนมัติ

from rasterio.features import rasterize
from sklearn.metrics import f1_score, jaccard_score, precision_score, recall_score
import pandas as pd

def evaluate_on_holdout(raster_path, model_path, architecture, encoder_name,
                         reference_vector_path, out_dir, num_channels=2, num_classes=2):
    
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(raster_path))[0]
    mask_path = os.path.join(out_dir, f"{base}_pred_mask.tif")
    proba_path = os.path.join(out_dir, f"{base}_pred_proba.tif")

    # 1. ให้ AI ทำนายบนภาพ (Prediction)
    # ฟังก์ชันนี้จะตัดภาพใหญ่เป็นแผ่นเล็กๆ (Sliding window) ให้ AI ทาย แล้วนำมาปะติดปะต่อกันเป็นภาพใหญ่โดยอัตโนมัติ
    geoai.semantic_segmentation(
        input_path=raster_path,        # ไฟล์ภาพ SAR ต้นฉบับ
        output_path=mask_path,         # ไฟล์ผลลัพธ์ (ภาพขาวดำ: 0=ไม่ใช่ปาล์ม, 1=ปาล์ม)
        model_path=model_path,         # ไฟล์โมเดลที่สอนเสร็จแล้ว (Best checkpoint)
        architecture=architecture,     # โครงสร้างโมเดล (เช่น SegFormer)
        encoder_name=encoder_name,     # โครงสร้างส่วนตา (เช่น ResNet-152)
        num_channels=num_channels,     # จำนวนแบนด์ (2 แบนด์ คือ VV, VH)
        num_classes=num_classes,       # จำนวนประเภท (2 ประเภท คือ พื้นที่ทั่วไป และ ปาล์ม)
        window_size=512,               # ตัดกรอบทีละ 512x512 พิกเซล ตอนทำนาย (ใหญ่กว่าตอนเทรนได้)
        overlap=256,                   # ระยะซ้อนทับกัน เพื่อไม่ให้เกิดรอยต่อตรงขอบภาพ (Smooth blending)
        batch_size=4,                  # ประมวลผลพร้อมกัน 4 กรอบ (ถ้าแรมการ์ดจอเยอะ ปรับเพิ่มได้)
        probability_path=proba_path,   # บันทึกภาพแบบ Probability Map (ค่า 0.0 - 1.0) ไว้ดูด้วย
    )

    # 2. เตรียม "คำเฉลย" (Ground Truth) เพื่อนำมาตรวจข้อสอบ
    # เปิดอ่านไฟล์ผลลัพธ์ที่ AI เพิ่งสร้าง เพื่อดึงค่าขนาดภาพ (กว้างxยาว) และพิกัดแผนที่ (crs, transform)
    with rasterio.open(mask_path) as src:
        pred = src.read(1)             # อ่านพิกเซลผลลัพธ์มาเก็บไว้ในตัวแปร pred
        transform = src.transform      # พิกัดตำแหน่งมุมภาพ (ใช้อ้างอิงสถานที่จริง)
        crs = src.crs                  # ระบบพิกัดอ้างอิง (เช่น EPSG:32647)
        shape = (src.height, src.width) # ขนาดความสูงและความกว้างของภาพ

    # โหลดไฟล์พิกัดรูปแปลงปาล์ม (GeoJSON) และปรับระบบพิกัดให้ตรงกับรูปดาวเทียม
    gdf_ref = gpd.read_file(reference_vector_path).to_crs(crs)
    
    # คำสั่ง rasterize: แปลงเส้นขอบเขตแปลง (Vector) ให้กลายเป็นภาพพิกเซลขาวดำ (Raster)
    gt = rasterize(
        [(geom, 1) for geom in gdf_ref.geometry], # ระบายสีเลข 1 ทับตรงบริเวณที่เป็นแปลงปาล์ม
        out_shape=shape,       # ขนาดภาพต้องเท่ากับรูปที่ AI ทำนาย
        transform=transform,   # พิกัดแผนที่ต้องตรงกัน
        fill=0,                # พื้นที่อื่นๆ ที่ไม่ใช่แปลงปาล์ม ให้เทสีเลข 0 ลงไป
        all_touched=True,      # ถ้าเส้นแปลงปาล์มแตะโดนพิกเซลไหนนิดเดียว ก็ให้นับเป็น 1 ทันที
        dtype="uint8",         # ชนิดข้อมูลตัวเลขขนาดเล็กเพื่อประหยัดหน่วยความจำ (RAM)
    )


    # 3. คำนวณคะแนน IoU, F1
    pred_flat = (pred.flatten() > 0).astype(np.uint8)
    gt_flat = gt.flatten()

    metrics = {
        "raster": base,
        "IoU": jaccard_score(gt_flat, pred_flat, average="binary", zero_division=0),
        "F1": f1_score(gt_flat, pred_flat, average="binary", zero_division=0),
        "Precision": precision_score(gt_flat, pred_flat, average="binary", zero_division=0),
        "Recall": recall_score(gt_flat, pred_flat, average="binary", zero_division=0),
    }
    return metrics, mask_path, proba_path

print("✅ โค้ดตรวจข้อสอบ (evaluate_on_holdout) พร้อมใช้งาน")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4.2 — รันตรวจข้อสอบบน Hold-out
# ═══════════════════════════════════════════════════════════

# ── หาภาพที่เป็น Hold-out ──
HOLDOUT_YEAR_MONTHS = {"2026_M05"}
holdout_raster_files = []
for f in sorted(glob.glob(os.path.join(RASTER_DIR, "Sentinel-1*.tif"))):
    m = re.search(r"(\d{4})_(M\d{2})", os.path.basename(f))
    if m and f"{m.group(1)}_{m.group(2)}" in HOLDOUT_YEAR_MONTHS:
        holdout_raster_files.append(f)

# ── เลือก model ที่ดีที่สุด ──
model_path = os.path.join(MODEL_OUTPUT, "best_model.pth")
if not os.path.exists(model_path):
    # ถ้าไม่มี best ให้เอา checkpoint ล่าสุดแทน
    checkpoints = sorted(glob.glob(os.path.join(MODEL_OUTPUT, "checkpoint_*.pth")))
    if checkpoints:
        model_path = checkpoints[-1]

print(f"🔵 ประเมินผลบนภาพ: {len(holdout_raster_files)} ไฟล์")
print(f"🧠 ใช้โมเดล: {os.path.basename(model_path)}")

holdout_eval_dir = os.path.join(OUTPUT_DIR, "holdout_eval")
results = []

for raster_path in holdout_raster_files:
    print(f"\n📡 กำลังรันทำนาย: {os.path.basename(raster_path)}")
    metrics, mask_path, proba_path = evaluate_on_holdout(
        raster_path=raster_path,
        model_path=model_path,
        architecture=ARCHITECTURE,
        encoder_name=ENCODER_NAME,
        reference_vector_path=VECTOR_PATH,
        out_dir=holdout_eval_dir,
    )
    results.append(metrics)

# ── แสดงตารางคะแนน ──
df = pd.DataFrame(results)
print("\n=== 🏆 สมุดพก: สรุปคะแนนสอบ ===")
df

---

## ขั้นตอนที่ 5 — Visualize ผลทำนาย

ดูตัวเลขอย่างเดียวอาจไม่เห็นภาพ เรามาเปิดภาพดูว่า AI ทายถูก (TP), ทายผิด (FP), ทายตกหล่น (FN) ตรงไหนบ้าง

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5.1 — แสดงภาพเปรียบเทียบ 4 ช่อง (TP/FP/FN)
# ═══════════════════════════════════════════════════════════

import matplotlib.patches as mpatches

def visualize_prediction(raster_path, mask_path, reference_vector_path, max_size=1500):
    base = os.path.splitext(os.path.basename(raster_path))[0]
    
    with rasterio.open(raster_path) as src:
        h, w = src.height, src.width
        scale = min(1.0, max_size / max(h, w))
        out_h, out_w = max(1, int(h * scale)), max(1, int(w * scale))
        sar_arr = src.read(1, out_shape=(out_h, out_w),
                          resampling=rasterio.enums.Resampling.nearest)
        transform = src.transform * src.transform.scale(w / out_w, h / out_h)
        crs = src.crs
    
    p2, p98 = np.nanpercentile(sar_arr, [2, 98])
    sar_stretched = np.clip((sar_arr - p2) / (p98 - p2 + 1e-6), 0, 1)
    
    with rasterio.open(mask_path) as src:
        pred = src.read(1, out_shape=(out_h, out_w),
                        resampling=rasterio.enums.Resampling.nearest)
        pred_bin = (pred > 0).astype(np.uint8)
    
    gdf_ref = gpd.read_file(reference_vector_path).to_crs(crs)
    gt = rasterize(
        [(geom, 1) for geom in gdf_ref.geometry],
        out_shape=(out_h, out_w), transform=transform,
        fill=0, all_touched=True, dtype="uint8",
    )
    
    # คำนวณความถูกต้องแต่ละ Pixel
    tp = (pred_bin == 1) & (gt == 1)   # True Positive: ถูกต้อง
    fp = (pred_bin == 1) & (gt == 0)   # False Positive: ทายเกิน (นึกว่าเป็นปาล์มแต่ไม่ใช่)
    fn = (pred_bin == 0) & (gt == 1)   # False Negative: พลาดไป (เป็นปาล์มแต่มองไม่เห็น)
    
    # ระบายสี
    overlay = np.zeros((out_h, out_w, 3), dtype=np.float32)
    overlay[tp] = [0.0, 0.8, 0.0]   # เขียว = ทายถูก
    overlay[fp] = [0.9, 0.1, 0.1]   # แดง = ทายเกิน
    overlay[fn] = [0.1, 0.3, 0.9]   # น้ำเงิน = พลาด
    bg = ~(tp | fp | fn)
    for c in range(3):
        overlay[..., c][bg] = sar_stretched[bg] * 0.6
    
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    axes[0].imshow(sar_stretched, cmap="gray")
    axes[0].set_title(f"ภาพ SAR ต้นฉบับ (VV)\n{base[:40]}")
    axes[0].axis("off")
    
    axes[1].imshow(gt, cmap="Greens")
    axes[1].set_title("คำเฉลย (Ground Truth)")
    axes[1].axis("off")
    
    axes[2].imshow(pred_bin, cmap="Oranges")
    axes[2].set_title("สิ่งที่ AI ทำนาย (Prediction)")
    axes[2].axis("off")
    
    axes[3].imshow(overlay)
    axes[3].set_title("เปรียบเทียบผล (Overlay)")
    axes[3].axis("off")
    
    legend_patches = [
        mpatches.Patch(color=[0.0, 0.8, 0.0], label="เขียว: ทายถูก (TP)"),
        mpatches.Patch(color=[0.9, 0.1, 0.1], label="แดง: ทายเกิน (FP)"),
        mpatches.Patch(color=[0.1, 0.3, 0.9], label="น้ำเงิน: ตกหล่น (FN)"),
    ]
    axes[3].legend(handles=legend_patches, loc="lower right", fontsize=10, framealpha=0.9)
    
    plt.tight_layout()
    plt.show()

# แสดงผลทุกภาพ holdout
for raster_path in holdout_raster_files:
    base = os.path.splitext(os.path.basename(raster_path))[0]
    mask_path = os.path.join(holdout_eval_dir, f"{base}_pred_mask.tif")
    if os.path.exists(mask_path):
        visualize_prediction(raster_path, mask_path, VECTOR_PATH)


---

## สรุปบทที่ 3

ยินดีด้วย! คุณได้สอน AI (Train Model) สำเร็จแล้ว ตอนนี้เรามีสมองกลที่ดูภาพ SAR แล้วรู้ว่าตรงน่าจะคือปาล์ม



### 📚 ไปต่อ → [บทที่ 4 — จากโมเดลสู่แผนที่จริง (Predict & Map)](04_predict_and_map.ipynb)